# 07 — Moteur B : classification supervisée (LightGBM)
Entraînement sur le train (toutes les fraudes + 1/20 des légitimes, poids 20),
arrêt et seuil réglés sur la validation (2015). Les tests A et B sont évalués une seule fois, à la fin.

In [ ]:
import sys
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

pd.set_option("display.max_columns", None)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from secumon.ml.features import gold
from secumon.ml.models import dataset

GOLD = ROOT / "data" / "processed" / "gold" / "gold.parquet"
TMP_LINUX = Path.home() / ".cache" / "secumon-duckdb-tmp"
TMP_LINUX.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET temp_directory = '{TMP_LINUX.as_posix()}'")
con.execute("SET memory_limit = '1GB'")   # lire un Parquet ne demande pas plus
con.execute("SET threads = 2")
con.execute("SET enable_progress_bar = false")

print("GOLD existe :", GOLD.exists())
print("lightgbm", lgb.__version__)

In [ ]:
t0 = time.perf_counter()
train = dataset.load_split(con, GOLD, "train", sample_every=dataset.LEGIT_SAMPLE_EVERY)
val = dataset.load_split(con, GOLD, "validation")
print(f"Chargement : {time.perf_counter() - t0:.0f} s")

categories = dataset.make_categories(train)
X_train, y_train, w_train = dataset.features_frame(train, categories), train["is_fraud"], train[dataset.WEIGHT]
X_val, y_val = dataset.features_frame(val, categories), val["is_fraud"]

for nom, df in [("train", train), ("validation", val)]:
    print(f"{nom:11s} : {len(df):>9,} lignes | {int(df['is_fraud'].sum()):>6,} fraudes | "
          f"{df.memory_usage(deep=True).sum() / 1e6:,.0f} Mo")
print("Poids total du train (equivalent transactions) :", f"{w_train.sum():,.0f}")

In [ ]:
params = {
    "objective": "binary",
    "metric": "average_precision",   # PR-AUC, adaptee aux classes tres desequilibrees
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "num_threads": 4,
    "seed": 42,
    "verbosity": -1,
}

train_set = lgb.Dataset(X_train, label=y_train, weight=w_train)
val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

t0 = time.perf_counter()
model = lgb.train(
    params, train_set, num_boost_round=2000,
    valid_sets=[val_set], valid_names=["validation"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)
print(f"Entrainement : {time.perf_counter() - t0:.0f} s | meilleure iteration : {model.best_iteration}")

In [ ]:
p_val = model.predict(X_val, num_iteration=model.best_iteration)

print(f"PR-AUC  validation : {average_precision_score(y_val, p_val):.4f}   "
      f"(hasard = taux de fraude = {y_val.mean():.4f})")
print(f"ROC-AUC validation : {roc_auc_score(y_val, p_val):.4f}")

prec, rec, thr = precision_recall_curve(y_val, p_val)
f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
best = int(np.argmax(f1))
SEUIL = float(thr[best])
alertes = int((p_val >= SEUIL).sum())

print(f"\nSeuil retenu (F1 max sur la validation) : {SEUIL:.4f}")
print(f"  precision : {prec[best]:.3f}  -> {prec[best]:.1%} des alertes sont de vraies fraudes")
print(f"  rappel    : {rec[best]:.3f}  -> {rec[best]:.1%} des fraudes sont detectees")
print(f"  alertes   : {alertes:,} sur {len(y_val):,} transactions ({alertes / len(y_val):.3%})")

In [ ]:
importance = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
importance["part_pct"] = (100 * importance["gain"] / importance["gain"].sum()).round(2)
importance.head(15)

In [ ]:
import json

MODELS = ROOT / "data" / "processed" / "models" / "engine_b"
MODELS.mkdir(parents=True, exist_ok=True)

model.save_model(str(MODELS / "lgbm.txt"), num_iteration=model.best_iteration)
meta = {
    "threshold": SEUIL,
    "best_iteration": model.best_iteration,
    "params": params,
    "features": gold.FEATURE_COLUMNS,
    "categories": categories,
    "legit_sample_every": dataset.LEGIT_SAMPLE_EVERY,
    "validation": {
        "pr_auc": float(average_precision_score(y_val, p_val)),
        "roc_auc": float(roc_auc_score(y_val, p_val)),
        "precision": float(prec[best]),
        "recall": float(rec[best]),
    },
    "lightgbm_version": lgb.__version__,
}
(MODELS / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))

# Verification : le modele recharge donne exactement les memes predictions.
recharge = lgb.Booster(model_file=str(MODELS / "lgbm.txt"))
echantillon = X_val.iloc[:5000]
identique = np.allclose(recharge.predict(echantillon), model.predict(echantillon, num_iteration=model.best_iteration))
print("Modele sauvegarde :", MODELS)
print("Rechargement identique :", identique)

## Évaluation finale — tests A et B (une seule fois, modèle et seuil figés)

In [ ]:
import gc
import importlib
import json

importlib.reload(dataset)                       # prend en compte iter_split

for nom in ["train", "val", "X_train", "X_val", "train_set", "val_set"]:
    globals().pop(nom, None)
gc.collect()

MODELS = ROOT / "data" / "processed" / "models" / "engine_b"
meta = json.loads((MODELS / "meta.json").read_text())
model = lgb.Booster(model_file=str(MODELS / "lgbm.txt"))
SEUIL = meta["threshold"]
categories = meta["categories"]
print(f"Modele recharge | seuil fige : {SEUIL:.4f} | iterations : {meta['best_iteration']}")
print("iter_split disponible :", hasattr(dataset, "iter_split"))

In [ ]:
t0 = time.perf_counter()
morceaux = []
for split in ["test_a", "test_b"]:
    for part in dataset.iter_split(con, GOLD, split, n_parts=12):
        part_scores = part[["tx_id", "split", "is_fraud", "merchant_country", "is_online"]].copy()
        part_scores["score"] = model.predict(dataset.features_frame(part, categories))
        morceaux.append(part_scores)
        del part
        gc.collect()
scores = pd.concat(morceaux, ignore_index=True)
del morceaux

scores["zone"] = np.where(scores["is_online"], "en ligne",
                 np.where(scores["merchant_country"] == "United States", "Etats-Unis",
                 np.where(scores["merchant_country"] == "Italy", "Italie", "etranger hors Italie")))
scores.to_parquet(MODELS / "scores_tests.parquet", index=False)

print(f"Scoring : {time.perf_counter() - t0:.0f} s")
print(scores.groupby("split").agg(transactions=("tx_id", "size"), fraudes=("is_fraud", "sum")))

In [ ]:
import json
MODELS = ROOT / "data" / "processed" / "models" / "engine_b"
SEUIL = json.loads((MODELS / "meta.json").read_text())["threshold"]
scores = pd.read_parquet(MODELS / "scores_tests.parquet")
print(len(scores), "lignes | seuil", round(SEUIL, 4))

In [ ]:
from sklearn.metrics import average_precision_score

def resume(df):
    y, p = df["is_fraud"].to_numpy(dtype=bool), df["score"].to_numpy()
    alerte = p >= SEUIL
    tp, n_f, n_a = int((alerte & y).sum()), int(y.sum()), int(alerte.sum())
    return pd.Series({
        "transactions": len(df),
        "fraudes": n_f,
        "pr_auc": average_precision_score(y, p) if 0 < n_f < len(df) else np.nan,
        "rappel": tp / n_f if n_f else np.nan,
        "precision": tp / n_a if n_a else np.nan,
        "alertes": n_a,
        "taux_alerte_pct": 100 * n_a / len(df),
    })

print("=== Global ===")
display(scores.groupby("split")[["is_fraud", "score"]].apply(resume).round(4))
print("=== Par zone ===")
display(scores.groupby(["split", "zone"])[["is_fraud", "score"]].apply(resume).round(4))

In [ ]:
ita = con.sql(f"""
    SELECT tx_id, country_new, merchant_new, mcc_new, dist_home_km, n_tx_24h, amount
    FROM '{GOLD}'
    WHERE split = 'test_b' AND merchant_country = 'Italy'
""").df().merge(scores[["tx_id", "is_fraud", "score"]], on="tx_id")

ita["groupe"] = np.where(~ita["is_fraud"], "1. legitime",
                np.where(ita["score"] >= SEUIL, "2. fraude detectee", "3. fraude manquee"))

ita.groupby("groupe").agg(
    transactions=("tx_id", "size"),
    pct_pays_nouveau=("country_new", "mean"),
    pct_commercant_nouveau=("merchant_new", "mean"),
    mediane_dist_km=("dist_home_km", "median"),
    mediane_n_tx_24h=("n_tx_24h", "median"),
    mediane_montant=("amount", "median"),
    mediane_score=("score", "median"),
).round(3)

## Conclusion — moteur B v1 (LightGBM, 2 000 arbres, seuil figé 0,7743)

| Jeu | PR-AUC | Rappel | Précision | Taux d'alerte |
|---|---|---|---|---|
| Validation (2015) | 0,896 | 81,7 % | 88,2 % | 0,179 % |
| **Test A — fraudes connues (2016)** | **0,837** | **73,6 %** | **87,4 %** | 0,176 % |
| **Test B — campagne inconnue (2017–2020)** | **0,498** | **41,1 %** | **66,0 %** | 0,055 % |
| dont Italie (4 682 fraudes, jamais vues) | 0,991* | 40,2 % | 99,9 % | — |

\* 81,5 % des transactions italiennes sont frauduleuses : le hasard ferait 0,815.

**Fraudes connues** : résultats stables entre validation et test A → pas de surapprentissage.

**Campagne inconnue** : 40 % détectés sans aucune règle de pays (contexte client uniquement),
précision 99,9 %. Les fraudes manquées (score médian 0,031) ont lieu chez des commerçants déjà
utilisés (25,5 % nouveaux contre 65 % pour les détectées) : `country_new` ne marque que la 1re fraude
de chaque carte (~9 fraudes par carte).

**Dérive de prévalence** : en ligne, la précision passe de 89 % (test A) à 10,9 % (test B) car
la proportion de fraudes chute de 1,4 % à 0,019 % → nécessité de surveiller et recalibrer le seuil.

**Limites** : 23 fraudes US dans le test B (non concluant) ; early stopping non atteint à 2 000 arbres.

**Suites** : moteur A (anomalies) pour les attaques nouvelles ; features d'épisode de voyage
(à évaluer en marquant le test B comme « après analyse d'erreurs »).